In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [20]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
torch.manual_seed(0)

batch_size = 128
block_size = 64
n_embd = 256

n_head = 4
n_kv_groups = 2

n_layer = 8
dropout = 0.0

learning_rate = 1e-3
epochs = 1000

eval_interval = 100
eval_iters = 200

cuda


# Data

In [3]:
from datasets import load_dataset

dataset = load_dataset("Jr23xd23/ArabicText-Large", split="train")
dataset = dataset.shuffle(seed=42).select(range(5000))

with open("samples.txt", "w", encoding="utf-8") as f:
    for i in range(len(dataset)):
        f.write(dataset[i]["text"] + "\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/251M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/173M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00003-of-00006.parquet:   0%|          | 0.00/156M [00:00<?, ?B/s]

data/train-00004-of-00006.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

data/train-00005-of-00006.parquet:   0%|          | 0.00/149M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/743288 [00:00<?, ? examples/s]

In [4]:
with open('samples.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

# Grouped Query Attention

In [15]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, emb_dim, n_heads, n_kv_groups, base=10000):
        super().__init__()

        assert n_heads % n_kv_groups == 0

        self.n_heads = n_heads
        self.n_kv_groups = n_kv_groups
        self.head_dim = emb_dim // n_heads
        self.group_size = n_heads // n_kv_groups

        self.base = base
        i = torch.arange(0, self.head_dim, 2)
        self.register_buffer("freq", 1.0 / (self.base ** (i / self.head_dim)))

        self.W_q = nn.Linear(emb_dim, self.head_dim * n_heads, bias=False)
        self.W_k = nn.Linear(emb_dim, self.head_dim * n_kv_groups, bias=False)
        self.W_v = nn.Linear(emb_dim, self.head_dim * n_kv_groups, bias=False)

        self.out_proj = nn.Linear(n_heads * self.head_dim, emb_dim, bias=False)

# ----------------------------------------------------------------------------------------------------------------------------------
    def compute_rope(self, x):
        batch, n_heads, tokens, head_dim = x.shape

        positions = torch.arange(tokens, device=x.device)
        angles = torch.outer(positions, self.freq.to(x.device))

        cos = torch.cos(angles)[None, None, :, :]
        sin = torch.sin(angles)[None, None, :, :]

        x = x.view(batch, n_heads, tokens, head_dim // 2, 2)

        x0 = x[..., 0]
        x1 = x[..., 1]

        x_rotated = torch.stack(
            [
                x0 * cos - x1 * sin,
                x0 * sin + x1 * cos,
            ],
            dim=-1
        )

        return x_rotated.flatten(-2)

# ----------------------------------------------------------------------------------------------------------------------------------
    def forward(self, x):
        batch, tokens, emb = x.shape

        Q = self.W_q(x).view(batch, tokens, self.n_heads, self.head_dim).transpose(1, 2)     # B,T,C @ C, (d_H*n_H)  --> B,T,(d_H*n_H)  --> B, T, n_H, d_H  -----> transpose: B, n_H, T, d_H
        K = self.W_k(x).view(batch, tokens, self.n_kv_groups, self.head_dim).transpose(1, 2) # B,T,C @ C, (d_H*n_kv) --> B,T,(d_H*n_kv) --> B, T, n_kv, d_H -----> transpose: B, n_kv, T, d_H
        V = self.W_v(x).view(batch, tokens, self.n_kv_groups, self.head_dim).transpose(1, 2) # نفسه

        Q = self.compute_rope(Q)
        K = self.compute_rope(K)

        K = K.repeat_interleave(self.group_size, dim=1)
        V = V.repeat_interleave(self.group_size, dim=1)

        s1 = (Q @ K.transpose(2, 3)) / (self.head_dim ** 0.5)  # (Q @ K.T) / dk ** 0.5
        # mask
        mask = torch.triu(torch.ones(tokens, tokens, device=x.device), diagonal=1).bool()

        s2 = s1.masked_fill(mask, float('-inf'))
        s3 = torch.softmax(s2, dim=-1) # softmax
        s4 = (s3 @ V).transpose(1, 2).contiguous()  # ... @ V

        out = s4.reshape(batch, tokens, self.n_heads * self.head_dim)

        return self.out_proj(out)

# FFN

In [7]:
class SwiGLU_FFN(nn.Module):
    def __init__(self, n_embd, beta=1.0):
        super().__init__()

        self.beta = beta
        self.W1 = nn.Linear(n_embd, int((8/3) * n_embd))
        self.W2 = nn.Linear(n_embd, int((8/3) * n_embd))
        self.out = nn.Linear(int((8/3) * n_embd), n_embd)

    def swish(self, x):
        return x * F.sigmoid(self.beta * x)

    def forward(self, x):
        x1 = self.W1(x)
        x2 = self.swish(self.W2(x))
        x = x1 * x2
        return self.out(x)

In [8]:
class GEGLU(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=-1)
        return x1 * F.gelu(x2)


class SimpleNet(nn.Module):
    def __init__(self, emb_size):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(emb_size, (4*emb_size)* 2)
        self.geglu = GEGLU()
        self.fc2 = nn.Linear(4*emb_size, emb_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.geglu(x)
        x = self.fc2(x)
        return x

# Norm

In [9]:
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.emb_dim = emb_dim
        self.weight = nn.Parameter(torch.ones(emb_dim)).float()

    def forward(self, x):
        means = x.pow(2).mean(dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(means + self.eps)
        return (x_normed * self.weight).to(dtype=x.dtype)

# ...

In [10]:
class Block(nn.Module):
    def __init__(self,emb_dim, n_heads, n_kv_groups):
        super().__init__()
        self.att = GroupedQueryAttention(emb_dim, n_heads, n_kv_groups)
        self.ffn = SwiGLU_FFN(n_embd)
        self.norm1 = RMSNorm(emb_dim)
        self.norm2 = RMSNorm(emb_dim)

    def forward(self, x):
        x = x + self.att(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

In [11]:
class Fanar(nn.Module):
  def __init__(self):
      super().__init__()
      self.token_emb = nn.Embedding(vocab_size, n_embd)
      self.blocks = nn.ModuleList([Block(n_embd, n_head, n_kv_groups) for _ in range(n_layer)])
      self.ln_f = RMSNorm(n_embd)
      self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
      B, T = idx.shape

      x = self.token_emb(idx) # (B,T,C)

      for block in self.blocks:
        x = block(x)

      x = self.ln_f(x)
      logits = self.lm_head(x)


      if targets is None:
          loss = None
      else:
          B, T, C = logits.shape
          logits = logits.view(B*T, C)
          targets = targets.view(B*T)
          loss = F.cross_entropy(logits, targets)

      return logits, loss

  def generate(self, idx, max_new_tokens):
      for _ in range(max_new_tokens):
           idx_cond = idx[:, -block_size:]
           logits, loss = self(idx_cond)
           logits = logits[:, -1, :] # becomes (B, C)
           probs = F.softmax(logits, dim=-1) # (B, C)
           idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
           idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
      return idx

In [17]:
model = Fanar().to(device)
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

6.521156 M parameters


In [18]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [21]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for epoch in range(epochs):

    if epoch % eval_interval == 0 or epoch == epochs - 1:
        losses = estimate_loss()
        print(f"step {epoch}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 1.9784, val loss 1.9749
step 100: train loss 1.8734, val loss 1.8674
step 200: train loss 1.7968, val loss 1.8008
step 300: train loss 1.7468, val loss 1.7476
step 400: train loss 1.7078, val loss 1.7023
step 500: train loss 1.6701, val loss 1.6735
step 600: train loss 1.6336, val loss 1.6408
step 700: train loss 1.6085, val loss 1.6127
step 800: train loss 1.5950, val loss 1.5975
step 900: train loss 1.5738, val loss 1.5871
step 999: train loss 1.5616, val loss 1.5729


In [23]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=2000)[0].tolist()))


مورباتيك (18 نيوانسين لديها ماري (والمسرمر) كوريا جدا، أحمد اكتشافه. حوالي 7/«كل 12ما (261) - أبرامج عام 2024-2034. طوع هيليناب نيزورتا في السنة الأهمي. ويعد الشرطة الفرنسية محمد الرأسية والاستوديوارت للمركبات المتجمعة، التي تتعمل على بلازمات الإباضية لتعامل عنلي إلى حمايتها في قرية الإله دورو الكبدية، ولا تهام بتخليص ساد واللاسلكي بالغدد للأساس على ألهماتها الثانيين رفض اللواء على يد نهر. سنة 135 ، حاجز المر قارئ في المتحاطة، فلا يصلح إثرها مع القدرة الملكية. ويووجد له علي جميكا مدير براوجينيا عصر الكادسني. البابيين (71 سبتمبر 1741)، (400 في نرويورك، بعدما تم تسخين الجمعة وكانت السلاحة تعني مموزن) معزولة من 5 أسبوع ل1914 كقدم فو ثاج من أي أمريكي) بمعروفا على الرجال. إن غير عشرة، صنف الضرم للسفن الخاصية الذي يدون اثنين بروتين وشامئ انحاض عام 1945 رجل نهر عالم آخر ماري إلى قائد "\مي في الدراستات وأويت، لكن الجوائز مطلوبين الطبق الأرز: أعاد جيمان (البلدات)، اليمرية النرويج. ليبانزون ألغاز تشارلز سيكس في الطلبة الجيارية. لكن عالم ثابت مجيم في نفوس وهجوبي فلسطين الكهرب لاهس بسهم. وتشاركل 